In [0]:
df_facilities = spark.sql(f"select * from regis_healthcare.silver.facilities;")
df_facilities.createOrReplaceTempView("facilities")

df_residents = spark.sql(f"select * from regis_healthcare.silver.residents;")
df_residents.createOrReplaceTempView("residents")

df_employees = spark.sql(f"select * from regis_healthcare.silver.employees;")
df_employees.createOrReplaceTempView("employees")

df_admissions = spark.sql(f"select * from regis_healthcare.silver.admissions;")
df_admissions.createOrReplaceTempView("admissions")

df_discharges = spark.sql(f"select * from regis_healthcare.silver.discharges;")
df_discharges.createOrReplaceTempView("discharges")

df_medications = spark.sql(f"select * from regis_healthcare.silver.medications;")
df_medications.createOrReplaceTempView("medications")

df_incidents = spark.sql(f"select * from regis_healthcare.silver.incidents;")
df_incidents.createOrReplaceTempView("incidents")

df_appointments = spark.sql(f"select * from regis_healthcare.silver.appointments;")
df_appointments.createOrReplaceTempView("appointments")

df_billing = spark.sql(f"select * from regis_healthcare.silver.billing;")
df_billing.createOrReplaceTempView("billing")

df_resident_feedback = spark.sql(f"select * from regis_healthcare.silver.resident_feedback;")
df_resident_feedback.createOrReplaceTempView("resident_feedback")

In [0]:
join_gold = spark.sql("""with cte1 as (
    select f.facility_id,
     a.resident_id,
f.facility_name,
f.address as address_fac,
  f.suburb as suburb_fac,
    f.state as state_fac,
      f.postcode as postcode_fac,
        f.phone as phone_fac,
          f.email as email_fac,
f.capacity as capacity_fac,
f.accreditation_status,
f.created_at as created_at_fac,
-- admissions table
 a.admission_id,
 a.admission_date,
 a.admission_type,
 a.referring_doctor as referring_doctor_adm ,
 a.ward as ward_adm,
 a.room_number as room_number_adm,
 a.status as status_adm,
 a.notes as notes_adm,
 a.created_at as created_at_adm
from facilities f join admissions a on f.facility_id = a.facility_id),
cte2 as(
    select c.*,
 r.first_name as first_name_res,
 r.last_name as last_name_res,
 r.date_of_birth as date_of_birth_res,
 r.gender as gender_res,
 r.address as address_res,
 r.suburb as suburb_res,
 r.state as state_res,
 r.postcode as postcode_res,
 r.phone as phone_res,
 r.email as email_res,
 r.care_level as care_level_res,
 r.medicare_number as medicare_number_res,
 r.created_at as created_at_res,
 r.updated_at as updated_at_res
 from cte1 c join residents r 
  on c.resident_id = r.resident_id
),
cte3 as(
    select p.*,d.discharge_id,
 d.discharge_date,
 d.discharge_reason,
 d.destination,
 d.authorized_by,
 d.notes as notes_disch,
 d.created_at as created_at_disch
 from cte2 as p join discharges d 
 on d.resident_id = p.resident_id
),
cte4 as (
    select c.*,
m.medication_id,
 m.medication_name,
 m.dosage as dosage_med,
 m.frequency as frequency_med,
 m.prescribing_doctor as prescribing_doctor_med,
 m.start_date as start_date_med,
 m.end_date as end_date_med,
 m.status as status_med,
 m.notes as notes_med,
 m.created_at as created_at_med
 from cte3 as c join medications as m 
 on c.resident_id = m.resident_id
),
cte5 as (
    select c.*,
  j.incident_id,
  j.incident_date,
  j.incident_type as incident_type_inc,
  j.severity as severity_inc,
  j.description as description_inc,
  j.reported_by as reported_by_inc,
  j.action_taken as action_taken_inc,
  j.follow_up_required as follow_up_required_inc,
  j.created_at as created_at_inc
 from cte4 as c join incidents as j 
 on c.resident_id = j.resident_id
),
cte6 as(
    select c.*,
 a.appointment_id,
 a.employee_id,
 a.appointment_date,
 a.appointment_type,
 a.status as status_app,
 a.location as location_app,
 a.notes as notes_app,
 a.created_at as created_at_app
 from cte5 as c join appointments as a
 on c.resident_id = a.resident_id
),
cte7 as(
    select c.*,
  e.first_name as first_name_emp,
  e.last_name as last_name_emp,
  e.job_title as job_title_emp,
  e.phone as phone_emp,
  e.email as email_emp,
  e.hire_date as hire_date_emp,
  e.employment_type,
  e.status as status_emp,
  e.salary as salary_emp,
  e.created_at as created_at_emp
 from cte6 as c join employees as e
 on c.employee_id = e.employee_id
),
cte8 as (select c.*,
 b.billing_id,
 b.billing_date,
 b.amount_aud as amount_aud_bill,
 b.billing_type,
 b.payment_status as payment_status_bill,
 b.payment_date,
 b.invoice_number as invoice_number_bill,
 b.notes as notes_bill,
 b.created_at as created_at_bill
 from cte7 as c join billing as b 
 on c.resident_id = b.resident_id
),
cte9 as (
    select c.*,
r.feedback_id,
 r.feedback_date,
 r.category as category_feed,
 r.rating as rating_feed,
 r.comment as comment_feed,
 r.submitted_by as submitted_by_feed,
 r.created_at as created_at_feed
 from cte8 as c join resident_feedback as r 
 on c.resident_id = r.resident_id
)
select * from cte9;""")
display(join_gold)


In [0]:
# %sql
# with cte1 as (
#     select f.facility_id,
# f.facility_name,
# f.address,
#   f.suburb,
#     f.state,
#       f.postcode,
#         f.phone,
#           f.email,
# f.capacity,
# f.accreditation_status,
# f.created_at,
# -- admissions table
#  a.admission_id,
#  a.resident_id,
#  a.admission_date,
#  a.admission_type,
#  a.referring_doctor,
#  a.ward,
#  a.room_number,
#  a.status,
#  a.notes,
#  a.created_at
# from facilities f join admissions a on f.facility_id = a.facility_id),
# cte2 as(
#     select c.*,
#  r.first_name,
#  r.last_name,
#  r.date_of_birth,
#  r.gender,
#  r.address,
#  r.suburb,
#  r.state,
#  r.postcode,
#  r.phone,
#  r.email,
#  r.facility_id,
#  r.care_level,
#  r.medicare_number,
#  r.created_at,
#  r.updated_at
#  from cte1 c join residents r 
#   on c.resident_id = r.resident_id
# ),
# cte3 as(
#     select p.*,d.discharge_id,
#  d.discharge_date,
#  d.discharge_reason,
#  d.destination,
#  d.authorized_by,
#  d.notes as notes_discharges,
#  d.created_at as created_at_discharges
#  from cte2 as p join discharges d 
#  on d.resident_id = p.resident_id
# ),
# cte4 as (
#     select c.*,
# m.medication_id,
#  m.medication_name,
#  m.dosage,
#  m.frequency,
#  m.prescribing_doctor,
#  m.start_date,
#  m.end_date,
#  m.status,
#  m.notes,
#  m.created_at
#  from cte3 as c join medications as m 
#  on c.resident_id = m.resident_id
# ),
# cte5 as (
#     select c.*,
#   j.incident_id,
#   j.facility_id,
#   j.incident_date,
#   j.incident_type,
#   j.severity,
#   j.description,
#   j.reported_by,
#   j.action_taken,
#   j.follow_up_required,
#   j.created_at
#  from cte4 as c join incidents as j 
#  on c.resident_id = j.resident_id
# ),
# cte6 as(
#     select c.*,
#  a.appointment_id,
#  a.employee_id,
#  a.appointment_date,
#  a.appointment_type,
#  a.status,
#  a.location,
#  a.notes,
#  a.created_at
#  from cte5 as c join appointments as a
#  on c.resident_id = a.resident_id
# ),
# cte7 as(
#     select c.*,
#   e.first_name,
#   e.last_name,
#   e.job_title,
#   e.facility_id,
#   e.phone,
#   e.email,
#   e.hire_date,
#   e.employment_type,
#   e.status,
#   e.salary,
#   e.created_at
#  from cte6 as c join employees as e
#  on c.employee_id = e.employee_id
# ),
# cte8 as (select c.*,
#  b.billing_id,
#  b.facility_id,
#  b.billing_date,
#  b.amount_aud,
#  b.billing_type,
#  b.payment_status,
#  b.payment_date,
#  b.invoice_number,
#  b.notes,
#  b.created_at
#  from cte7 as c join billing as b 
#  on c.resident_id = b.resident_id
# ),
# cte9 as (
#     select c.*,
# r.feedback_id,
#  r.resident_id,
#  r.facility_id,
#  r.feedback_date,
#  r.category,
#  r.rating,
#  r.comment,
#  r.submitted_by,
#  r.created_at
#  from cte8 as c join resident_feedback as r 
#  on c.resident_id = r.resident_id
# )
# select * from cte9;

In [0]:
# df_facilities.columns
# df_residents.columns
# df_employees.columns
# df_admissions.columns
# df_discharges.columns
# df_medications.columns
# df_incidents.columns
# df_appointments.columns
# df_billing.columns
# df_resident_feedback.columns

In [0]:
# df_facilities.columns
print(df_facilities.count())
# df_residents.columns
print(df_residents.count())
# df_employees.columns
print(df_employees.count())
# df_admissions.columns
print(df_admissions.count())
# df_discharges.columns
print(df_discharges.count())
# df_medications.columns
print(df_medications.count())
# df_incidents.columns
print(df_incidents.count())
# df_appointments.columns
print(df_appointments.count())
# df_billing.columns
print(df_billing.count())
# df_resident_feedback.columns
print(df_resident_feedback.count())
# 200
# 100000
# 10000
# 150000
# 80000
# 500000
# 50000
# 200000
# 300000
# 50000

# join table

In [0]:
# s3 loading
join_gold.write.format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .mode("overwrite")\
        .save("s3://regis-healthcare/parent-gold/")